# Testing Balancing Strategies

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE

In [6]:
# 1. Loading the data
train_df = pd.read_csv('trainn.csv')
val_df = pd.read_csv('vall.csv')

# Drop any accidental missing values
train_df = train_df.dropna(subset=['title'])
val_df = val_df.dropna(subset=['title'])

In [7]:
# 2. Convert text into numbers (TF-IDF) so the test model can read it
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train = vectorizer.fit_transform(train_df['title'])
y_train = train_df['label']

X_val = vectorizer.transform(val_df['title'])
y_val = val_df['label']

print("--- Testing Balancing Strategies ---")

# --- STRATEGY 1: Class Weights ---
# This tells the model to pay extra attention to the rare classes
model_weights = LogisticRegression(class_weight='balanced', max_iter=1000)
model_weights.fit(X_train, y_train)
preds_weights = model_weights.predict(X_val)

# We use 'macro' F1 to ensure it treats all three classes equally
f1_weights = f1_score(y_val, preds_weights, average='macro')
print(f"Strategy 1 (Class Weights) Validation F1 Score: {f1_weights:.4f}")

# --- STRATEGY 2: SMOTE ---
# This creates fake, realistic data points for the rare classes to balance the scales
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)
preds_smote = model_smote.predict(X_val)

f1_smote = f1_score(y_val, preds_smote, average='macro')
print(f"Strategy 2 (SMOTE) Validation F1 Score: {f1_smote:.4f}")

# --- The Conclusion ---
print("\n--- Final Conclusion ---")
if f1_weights > f1_smote:
    print("Use class_weight='balanced', it scored higher!")
else:
    print("Use SMOTE to balance the training data, it scored higher!")

--- Testing Balancing Strategies ---
Strategy 1 (Class Weights) Validation F1 Score: 0.9885
Strategy 2 (SMOTE) Validation F1 Score: 0.9897

--- Final Conclusion ---
Use SMOTE to balance the training data, it scored higher!
